# Word2Vec using Gensim

In [ ]:
import subprocess
import sys
import os
import csv

# --- AUTO-INSTALLER BLOCK ---
def maintain_dependencies():
    required_libraries = ['numpy', 'scipy', 'gensim']
    for lib in required_libraries:
        try:
            __import__(lib)
        except ImportError:
            print(f"📦 Library '{lib}' not found. Installing now...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

maintain_dependencies()
# ----------------------------

import numpy as np
from scipy.stats import spearmanr, pearsonr
from gensim.models import Word2Vec, FastText

# --- COSINE SIMILARITY ---
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity: (A · B) / (||A|| × ||B||)"""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

# --- CLASSIFICATION METRICS ---
def confusion_matrix_np(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tn, fp, fn, tp

def accuracy_np(tp, tn, fp, fn):
    total = tp + tn + fp + fn
    return (tp + tn) / total if total > 0 else 0.0

def precision_np(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall_np(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1_np(precision, recall):
    return 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# --- FILE LOADING ---
def load_text_file(filepath):
    """Reads a .txt file and returns a list of tokenized sentences."""
    sentences = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                tokens = line.lower().strip().split()
                if tokens:
                    sentences.append(tokens)
        return sentences
    except Exception as e:
        print(f"❌ Error reading {filepath}: {e}")
        return []

# --- CREATE SAMPLE CORPUS ---
def create_sample_corpus(filepath):
    """Creates a sample isiZulu corpus for testing"""
    sample_corpus = """umfazi nendoda bahamba esikoleni
ingane idla ukudla kwayo
inja ikati zidlala eyadini
isikole isikhungo semfundo
ikhaya indlu yomndeni
umfula ulwandle amanzi
uthisha umfundi bafunda
isitsha indishi kudla
ibhola umdlalo imidlalo
umuntu ubuntu ubuntu
itheku idolobha amadolobha
incwadi iphepha ukubhala
ikhompiyutha ikhibhodi theknoloji
indiza imoto isitimela ukuhamba
ucingo ukuxhumana uxhumano
umabonakude umsakazo ezindaba
abezindaba umsakazo ukubika
udokotela umhlengikazi ukwelapha
solwazi umfundi ukufunda
inkampani amasheya ukuhweba
isitoko indali ukuthenga
ibhange imali ukonga
ukhuni ihlathi amahlathi
inkosi indlovukazi umbuso
umbhishobhi uRabi unkulunkulu
inyoni iqhude izilwane
ithuluzi ukusebenza
umfana mfowethu umndeni
uhambo imoto ukuhamba
imali idola ingcebo impahla
imali ibhange ukufaka ukuhoxa ukuwasha
ihlosi isilwane izilwane i-zoo
usho njalo isikhathi
uqhawekazi mdikane isibindi
umphiko indiza
usuku ubusuku isikhathi
inzondo ucansi
isinkwa ibhotela ukudla
ikhukhamba izambane imifino
hlakaniphile isilima ukuhlakanipha
ukuzala iqanda
umtapo wezincwadi incwadi
igwaba inkosi
qalisa ithuluzi ukusebenza
ukuhlukumeza isidakamizwa
imeya inkosi amadolobha
isikweletu imali ukuboleka
umasipala uhulumeni isikhungo
inkohlakalo icala ububi
inyuvesi isikole ukufunda
isivivinyo u-matric ukuhlola
ingoma umculo icwecwe
umrepha umculi umculo
ikhwaya umbhalo amaculo
idume izindondo udumo
amaphoyisa abasolwa icala
isibhamu inhlamvu udubula
ubunhloli umkhondo uphenyisiso
isiteshi inkantolo amaphoyisa
ilokishi idolobha indawo
ihhotela isivakashi ukulala
ingqalasizinda ukuthuthukiswa ukwakha
emakhaya iphesheya ezweni
itekisi imoto ukuhamba
ubudokotela udokotela ukwelapha
isifo impilo ukugula
ubumnandi ukujabula injabulo
"""
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(sample_corpus)
    print(f"✅ Sample corpus created: {filepath}")


# ============================================================
# --- SYNONYM / SIMILAR WORD LOOKUP ---
# ============================================================

def get_word_synonyms(model, word, topn=5):
    """
    Returns the top-N most similar words (synonyms) for a single word.
    FastText handles OOV words via subword n-grams.
    """
    word = word.lower().strip()
    try:
        similar = model.wv.most_similar(word, topn=topn)
        in_vocab = word in model.wv.key_to_index
        return similar, in_vocab
    except Exception as e:
        return None, False


def get_sentence_synonyms(model, sentence, topn=3):
    """
    For each word in a sentence, finds its top synonyms.
    Returns a list of (original_word, [(synonym, score), ...]) tuples.
    Also returns a 'synonym sentence' by replacing each word with its top synonym.
    """
    tokens = sentence.lower().strip().split()
    results = []
    synonym_tokens = []

    for token in tokens:
        similar, in_vocab = get_word_synonyms(model, token, topn=topn)
        if similar:
            results.append((token, similar, in_vocab))
            synonym_tokens.append(similar[0][0])  # best synonym
        else:
            results.append((token, [], False))
            synonym_tokens.append(token)  # keep original if no synonym

    synonym_sentence = ' '.join(synonym_tokens)
    return results, synonym_sentence


def interactive_synonym_lookup(model):
    """
    Interactive loop: user inputs a word or sentence and receives synonyms.
    Type 'quit' or 'exit' to stop.
    """
    print("\n" + "="*70)
    print("🔍 ISIZULU SYNONYM LOOKUP — Powered by FastText Subword N-grams")
    print("="*70)
    print("📌 How to use:")
    print("   • Enter a single word  → get top synonyms for that word")
    print("   • Enter a sentence     → get synonyms for each word + a")
    print("                            paraphrased synonym sentence")
    print("   • Type 'quit' or 'exit' to stop\n")

    while True:
        try:
            user_input = input("➤  Enter a word or sentence: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Exiting synonym lookup. Goodbye!")
            break

        if not user_input:
            print("   ⚠️  Please enter something.\n")
            continue

        if user_input.lower() in ('quit', 'exit'):
            print("👋 Goodbye!")
            break

        tokens = user_input.split()

        # ── Single word ──────────────────────────────────────────────────
        if len(tokens) == 1:
            word = tokens[0]
            similar, in_vocab = get_word_synonyms(model, word, topn=5)
            vocab_label = "✅ In vocabulary" if in_vocab else "🔠 Computed via n-grams (OOV)"

            print(f"\n  Word      : '{word}'")
            print(f"  Status    : {vocab_label}")

            if similar:
                print(f"  Top synonyms / similar words:")
                for rank, (syn, score) in enumerate(similar, 1):
                    bar = "█" * int(score * 20)
                    print(f"    {rank}. {syn:<25} similarity: {score:.4f}  {bar}")
            else:
                print("  ❌ Could not retrieve synonyms for this word.")

        # ── Sentence (multiple words) ────────────────────────────────────
        else:
            results, synonym_sentence = get_sentence_synonyms(model, user_input, topn=3)

            print(f"\n  Original sentence  : \"{user_input}\"")
            print(f"  Synonym sentence   : \"{synonym_sentence}\"")
            print(f"\n  Word-by-word breakdown:")
            print(f"  {'Word':<25} {'Status':<18} {'Top Synonyms'}")
            print(f"  " + "-"*65)

            for word, syns, in_vocab in results:
                vocab_label = "In vocab" if in_vocab else "N-gram OOV"
                if syns:
                    syns_str = ", ".join([f"{s}({sc:.2f})" for s, sc in syns])
                else:
                    syns_str = "(none found)"
                print(f"  {word:<25} {vocab_label:<18} {syns_str}")

        print()  # blank line between queries


# ============================================================
# --- MAIN EXECUTION ---
# ============================================================

if __name__ == "__main__":
    CORPUS_FILE = 'isizulu_corpus.txt'
    OUTPUT_CSV = 'isizulu_fasttext_ngram_results.csv'

    # Test pairs
    isi_test_pairs = [
        ('inkosi', 'imeya', 8.45),
        ('imali', 'isikweletu', 7.12),
        ('uhulumeni', 'umasipala', 8.90),
        ('inkohlakalo', 'icala', 7.50),
        ('isikole', 'inyuvesi', 8.20),
        ('umfundi', 'uthisha', 7.65),
        ('izifundo', 'imiphumela', 6.80),
        ('u-matric', 'isivivinyo', 9.10),
        ('ingoma', 'icwecwe', 8.55),
        ('umrepha', 'umculi', 9.25),
        ('ikhwaya', 'umbhalo', 4.10),
        ('idume', 'izindondo', 6.40),
        ('amaphoyisa', 'abasolwa', 8.15),
        ('isibhamu', 'inhlamvu', 9.40),
        ('ubunhloli', 'umkhondo', 8.70),
        ('isiteshi', 'inkantolo', 6.95),
        ('idolobha', 'ilokishi', 7.30),
        ('isivakashi', 'ihhotela', 8.85),
        ('ingqalasizinda', 'ukuthuthukiswa', 7.75),
        ('emakhaya', 'iphesheya', 3.20),
        ('itekisi', 'ubudokotela', 1.15),
        ('umculo', 'isifo', 0.90),
        ('u-matric', 'ubumnandi', 2.50),
        ('inkosi', 'igwaba', 1.05),
    ]

    # Create sample corpus if it doesn't exist
    if not os.path.exists(CORPUS_FILE):
        print("📝 Creating sample corpus file...")
        create_sample_corpus(CORPUS_FILE)

    print(f"\n📂 Loading corpus from {CORPUS_FILE}...")
    sentences = load_text_file(CORPUS_FILE)

    if not sentences:
        print("🛑 The text file is empty.")
        sys.exit(1)

    print(f"✅ Loaded {len(sentences)} sentences")

    print(f"\n🚀 Training FastText model with subword n-grams...")
    print(f"   Architecture: Skip-gram with character n-grams")
    print(f"   Parameters: vector_size=200, window=7, n-grams=3-6, epochs=100\n")
    sys.stdout.flush()

    model = FastText(
        sentences=sentences,
        vector_size=200,
        window=7,
        min_count=2,
        epochs=100,
        sg=1,
        workers=4,
        alpha=0.025,
        min_alpha=0.0001,
        negative=10,
        sample=1e-1,
        min_n=3,
        max_n=6,
        word_ngrams=1,
        bucket=2000000
    )

    print(f"✅ FastText model trained! Vocabulary size: {len(model.wv)}")
    print(f"   Subword n-grams: {model.wv.min_n}-{model.wv.max_n} characters\n")
    sys.stdout.flush()

    # --- EVALUATION ---
    print("="*90)
    print("CALCULATING COSINE SIMILARITIES (WITH SUBWORD N-GRAMS)")
    print("="*90)

    cosine_scores = []
    human_scores = []
    results = []

    print(f"{'Word 1':<25} {'Word 2':<25} {'Human':<10} {'Cosine':<10} {'Status':<15}")
    print("-"*90)

    for w1, w2, h_score in isi_test_pairs:
        try:
            vec1 = model.wv[w1]
            vec2 = model.wv[w2]
            cos_sim = cosine_similarity(vec1, vec2)

            in_vocab_w1 = w1 in model.wv.key_to_index
            in_vocab_w2 = w2 in model.wv.key_to_index

            if in_vocab_w1 and in_vocab_w2:
                status = "In Vocab"
            elif in_vocab_w1 or in_vocab_w2:
                status = "Partial N-gram"
            else:
                status = "Full N-gram"

            print(f"{w1:<25} {w2:<25} {h_score:<10.2f} {cos_sim:<10.6f} {status:<15}")

            cosine_scores.append(cos_sim)
            human_scores.append(h_score)
            results.append({
                'word1': w1, 'word2': w2,
                'human_score': h_score,
                'cosine_similarity': cos_sim,
                'in_vocab': status
            })
        except Exception as e:
            print(f"{w1:<25} {w2:<25} {h_score:<10.2f} {'ERROR':<10} {str(e):<15}")

    print("="*90)
    sys.stdout.flush()

    if len(cosine_scores) >= 2:
        rho, rho_p = spearmanr(human_scores, cosine_scores)
        pear, pear_p = pearsonr(human_scores, cosine_scores)

        human_median = np.median(human_scores)
        cosine_median = np.median(cosine_scores)
        y_true = (np.array(human_scores) >= human_median).astype(int)
        y_pred = (np.array(cosine_scores) >= cosine_median).astype(int)

        tn, fp, fn, tp = confusion_matrix_np(y_true, y_pred)
        accuracy  = accuracy_np(tp, tn, fp, fn)
        precision = precision_np(tp, fp)
        recall    = recall_np(tp, fn)
        f1        = f1_np(precision, recall)

        print(f"\n📊 CORRELATION METRICS:")
        print(f"   Spearman: {rho:.6f} (p={rho_p:.6f})   Pearson: {pear:.6f} (p={pear_p:.6f})")
        print(f"\n📈 CLASSIFICATION METRICS:")
        print(f"   Accuracy: {accuracy:.6f}  Precision: {precision:.6f}  Recall: {recall:.6f}  F1: {f1:.6f}")
        print(f"\n🔍 CONFUSION MATRIX:  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
        print(f"\n📝 Coverage: {len(cosine_scores)}/{len(isi_test_pairs)} pairs ({len(cosine_scores)/len(isi_test_pairs)*100:.1f}%)")

        # Save CSV
        with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=['word1','word2','human_score','cosine_similarity','in_vocab'])
            writer.writeheader()
            writer.writerows(results)
        print(f"\n✅ Results saved to '{OUTPUT_CSV}'")

    # ============================================================
    # --- INTERACTIVE SYNONYM LOOKUP (new feature) ---
    # ============================================================
    interactive_synonym_lookup(model)


📂 Loading corpus from isizulu_corpus.txt...
✅ Loaded 11628 sentences

🚀 Training FastText model with subword n-grams...
   Architecture: Skip-gram with character n-grams
   Parameters:
      - Vector size: 200
      - Window: 7
      - Min n-gram: 3 (trigrams)
      - Max n-gram: 6 (hexagrams)
      - Epochs: 100
      - Min count: 2
   This enables handling of OOV words and morphological variations!

✅ FastText model trained! Vocabulary size: 5379
   Subword n-grams: 3-6 characters
   This model can now handle unseen words!

CALCULATING COSINE SIMILARITIES (WITH SUBWORD N-GRAMS)
Word 1                    Word 2                    Human      Cosine     Status         
------------------------------------------------------------------------------------------
inkosi                    imeya                     8.45       0.293793   In Vocab       
imali                     isikweletu                7.12       0.219082   Partial N-gram 
uhulumeni                 umasipala                